# Majority-Class Baseline

This notebook establishes a simple baseline for binary network intrusion detection using the NSL-KDD dataset. It loads the training and test files, labels normal traffic as `0` and attacks as `1`, excludes the `difficulty` metadata column, and applies the preprocessor fitted on the training data.

A scikit-learn `DummyClassifier` with the `most_frequent` strategy is fitted on the training set and evaluated on the test set. The notebook reports accuracy, precision, recall, F1-score, a classification report, and a confusion matrix, then saves the summary metrics to `results/metrics/majority_baseline.json` for comparison with later models.

Because this classifier always predicts the majority class in the training set (normal traffic), it serves as a simple reference point rather than a useful intrusion detector. In the current run, it achieves **43.07% accuracy** but **0.00 recall and F1-score for attacks**, demonstrating why accuracy alone is insufficient for evaluating intrusion detection. The training and test sets also have different class distributions, which is relevant when interpreting the results.

> **Update (shared harness):** The original analysis cells are kept unchanged, except the metrics-saving cell (path fix). The final section re-runs the same `DummyClassifier(strategy="most_frequent")` through the shared experiment harness. It uses the same seed, the same `train_sub`/`val` split and the same KDDTest+ evaluation as every other model, writes the unified `results/metrics/majority_baseline.json` and predictions, and logs an MLflow run. The baseline has no hyperparameters, so it gets no Optuna tuning; its threshold is fixed at 0.5.

In [2]:
import pandas as pd
import joblib
from pathlib import Path

# Resolve project paths whether the kernel starts in the project root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dataset" / "raw").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "dataset" / "raw"

columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted",
    "num_root", "num_file_creations", "num_shells", "num_access_files",
    "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate",
    "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate",
    "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "attack", "difficulty"
]

train_df = pd.read_csv(DATA_DIR / "KDDTrain+.csv", header=None, names=columns)
test_df = pd.read_csv(DATA_DIR / "KDDTest+.csv", header=None, names=columns)

# Separate labels from input features
X_train_raw = train_df.drop(columns=["attack", "difficulty"])
X_test_raw = test_df.drop(columns=["attack", "difficulty"])

y_train = (train_df["attack"] != "normal").astype(int)
y_test = (test_df["attack"] != "normal").astype(int)

# Load the preprocessor fitted on training data
preprocessor = joblib.load(PROJECT_ROOT / "models" / "preprocessor.joblib")

X_train = preprocessor.transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

In [3]:
from sklearn.dummy import DummyClassifier

# Always predict the most frequent class in the training labels
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

y_pred = baseline.predict(X_test)

print("Predicted classes:", set(y_pred))

Predicted classes: {np.int64(0)}


In [4]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Calculate metrics on the official test set
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred, zero_division=0))

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Normal", "Attack"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.43073237812181164
Precision: 0.0
Recall: 0.0
F1-score: 0.0

Classification report:
              precision    recall  f1-score   support

      Normal       0.43      1.00      0.60      9710
      Attack       0.00      0.00      0.00     12833

    accuracy                           0.43     22543
   macro avg       0.22      0.50      0.30     22543
weighted avg       0.19      0.43      0.26     22543

Confusion matrix:
[[ 9710     0]
 [12833     0]]


In [5]:
# updated: the legacy metrics write used a cwd-relative path ("../results/metrics") and an
# ad-hoc schema. Metrics are now written in the unified schema by the shared harness (final section).
legacy_metrics = {
    "model": "Majority-Class Baseline",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0)
}
print(legacy_metrics)

{'model': 'Majority-Class Baseline', 'accuracy': 0.43073237812181164, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}


## Baseline in the shared harness (unified metrics, MLflow, artifacts)

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src import experiment as ex
from src.features import build_pipeline

ex.init()                                   # seeds Python / NumPy / hash seed from config.yaml
train, test, novel = ex.load_data()
idx_tr, idx_va = ex.get_splits(train)       # identical rows to every other model
MODEL = "majority_baseline"

def make_pipeline(params):
    # Same preprocessing front-end as the tree models (irrelevant for a constant predictor,
    # kept so every saved pipeline accepts the same 41 raw features).
    return build_pipeline(DummyClassifier(strategy="most_frequent", random_state=ex.SEED), "tree")

run_id = ex.start_run(MODEL, {"family": "tree", "note": "no tuning (no hyperparameters)"})
res = ex.fit_and_evaluate(MODEL, run_id, make_pipeline, {"strategy": "most_frequent"}, train, test,
                          idx_tr, idx_va, novel, tune_threshold=False, export_onnx=False)
import pandas as pd
pd.DataFrame({k: res["metrics"][k] for k in ["train", "val", "test"]}).T[
    ["accuracy", "precision", "recall", "f1", "f1_macro", "roc_auc", "pr_auc", "fpr", "fnr"]].round(4)

The baseline always predicts *normal*, the training majority. On KDDTest+ that gives about 43% accuracy, 0 attack recall and macro-F1 ≈ 0.30, with ROC-AUC = PR-AUC baseline (0.5 / attack prevalence). Any useful detector must clear this floor by a wide margin. That is why accuracy alone is not used to compare models.